# AIRPATH-AI Milestone 3B — road network and segment ETA

This notebook reproduces the bounded OpenStreetMap route/ETA example.

Scope safeguards:

- routes remain inside the HealthyAir stations 2–6 support polygon;
- supported modes are walking and motorbike only;
- travel time uses explicit constant speeds, not traffic prediction;
- candidate routes are not ranked by PM2.5;
- no spatial estimator or exposure calculation is called;
- exact segment ETAs do not imply minute-level PM2.5 support.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eta_engine import DEFAULT_MODE_SPEED_KMH
from src.road_network import load_network, road_type_counts
from src.route_candidates import generate_milestone_outputs

In [ ]:
network_path = PROJECT_ROOT / "data/processed/road_network/healthyair_pilot_osm.json.gz"
network = load_network(network_path)

{
    "osm_snapshot_timestamp": network.metadata["osm_snapshot_timestamp"],
    "all_nodes": len(network.nodes),
    "all_directed_edges": len(network.edges),
    "walking": network.mode_counts("walking"),
    "motorbike": network.mode_counts("motorbike"),
    "speed_assumptions_kmh": DEFAULT_MODE_SPEED_KMH,
}

In [ ]:
outputs = generate_milestone_outputs(
    network_path,
    processed_directory=PROJECT_ROOT / "data/processed/road_network",
    report_root=PROJECT_ROOT / "reports",
    k=5,
)
outputs["route_summary"]

In [ ]:
motorbike_segments = outputs["segment_table"].loc[
    outputs["segment_table"]["route_id"].eq("motorbike-1")
]
display(motorbike_segments.head(8))
display(motorbike_segments.tail(2))

# This is the future route-to-spatial contract only. It is not passed to
# estimate_pm25 during Milestone 3B.
outputs["spatial_targets"][:3]

In [ ]:
display(Image(filename=PROJECT_ROOT / "reports/figures/road_network_pilot.png"))
display(Image(filename=PROJECT_ROOT / "reports/figures/road_network_candidate_routes.png"))